# Week 39

In [ ]:
!pip install -q --upgrade transformers datasets sacrebleu rouge_score evaluate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 4.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 34.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 21.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu12 25.6.0 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 22.0.0 which is incompatible.
pylibcudf-cu12 25.6.0 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 22.0.0 which is incompatible.


In [ ]:
import pandas as pd
import re
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt
import nltk
import transformers
import os
from nltk.util import ngrams
import math
nltk.download('punkt')
nltk.download('punkt_tab')
from nltk import FreqDist, ConditionalFreqDist
import torch
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from torch import nn
from tqdm.notebook import tqdm
from torch.utils.data import DataLoader
from torch.optim import AdamW
from typing import List, Tuple
from transformers import MarianMTModel, MarianTokenizer, DistilBertTokenizerFast
from tqdm import tqdm
import evaluate
from transformers import Seq2SeqTrainingArguments
from sacrebleu.metrics import BLEU
from transformers import pipeline
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Seq2SeqTrainer, DataCollatorForSeq2Seq
from transformers import MBartForConditionalGeneration, MBart50TokenizerFast
from google.colab import drive
drive.mount('/content/drive')


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


Mounted at /content/drive


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("google/mt5-small")
model = AutoModelForSeq2SeqLM.from_pretrained("google/mt5-small")

In [ ]:

# device = "cuda" if torch.cuda.is_available() else "cpu"

# Load multilingual mBART50 model
# trans_model_name = "facebook/mbart-large-50-many-to-many-mmt"
# trans_tokenizer = AutoTokenizer.from_pretrained(trans_model_name)
# trans_model = AutoModelForSeq2SeqLM.from_pretrained(trans_model_name).to(device)

In [ ]:
LANG_MAP = {
    "te": "te_IN",
    "ar": "ar_AR",
    "ko": "ko_KR"
}
TARGET_LANG = "te_IN"

def translate_to_telugu_mbart(text, source_lang="en_XX"):
    tokenizer.src_lang = source_lang
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True).to(device)

    translated_tokens = trans_model.generate(
        **inputs,
        forced_bos_token_id=tokenizer.lang_code_to_id[TARGET_LANG],
        max_length=64,
        num_beams=5,
        early_stopping=True
    )

    translated_text = tokenizer.batch_decode(translated_tokens, skip_special_tokens=True)[0]
    return translated_text

output_path = "/content/drive/MyDrive/translated_data/train_with_telugu.csv"

if os.path.exists(output_path):
    print(f"Found existing translations {output_path}")
    df_train_te = pd.read_csv(output_path)
else:
    print("No translations found, running translate_to_telugu_mbart")

    df_train_te = df_train_te[df_train_te['answer'].notnull()].reset_index(drop=True)

    tqdm.pandas(desc="Translating English answers to Telugu")
    df_train_te['answer_inlang_trans'] = df_train_te['answer'].progress_apply(
        lambda x: translate_to_telugu_mbart(x)
    )
    df_train_te['answer_inlang'] = df_train_te['answer_inlang'].combine_first(df_train_te['answer_inlang_trans'])
    df_train_te = df_train_te.drop(columns=['answer_inlang_trans'])

    df_train_te.to_csv(output_path, index=False, encoding="utf-8-sig")
    print(f"Translations saved to {output_path}")

val_output_path = "/content/drive/MyDrive/translated_data/val_with_telugu.csv"

if os.path.exists(val_output_path):
    print(f"Loading existing validation translations from {val_output_path}...")
    df_val_te = pd.read_csv(val_output_path)
else:
    print("Translating validation set...")
    tqdm.pandas(desc="Translating English answers to Telugu")
    df_val_te['answer_inlang_trans'] = df_val_te['answer'].progress_apply(
        lambda x: translate_to_telugu_mbart(x)
    )
    df_val_te['answer_inlang'] = df_val_te['answer_inlang'].combine_first(df_val_te['answer_inlang_trans'])
    df_val_te = df_val_te.drop(columns=['answer_inlang_trans'])
    df_val_te.to_csv(val_output_path, index=False, encoding="utf-8-sig")
    print(f"Saved translated validation set to {val_output_path}")

## Question + Context

In [ ]:
def get_question_context(row):
    question = row['question']
    context = row['context']
    return question + " " + context

df_train_te['input_text'] = df_train_te.apply(get_question_context, axis=1)
df_val_te['input_text'] = df_val_te.apply(get_question_context, axis=1)

df_train_te = df_train_te[df_train_te['answerable'] == True]

def get_target_language(row):
  answer_inlang = row['answer_inlang']
  return answer_inlang

def get_target_language_trans(row):
  answer_inlang_trans = row['answer_inlang_trans']
  return answer_inlang_trans

df_train_te['target_text'] = df_train_te.apply(get_target_language, axis=1)
df_val_te['target_text'] = df_val_te.apply(get_target_language_trans, axis=1)

In [ ]:
df_train_te = df_train_te[['input_text', 'target_text', 'answerable']]

df_val_te = df_val_te[['input_text', 'target_text', 'answerable']]

train_dataset = Dataset.from_pandas(df_train_te, preserve_index=False)
val_dataset = Dataset.from_pandas(df_val_te, preserve_index=False)

max_input_length = 768
max_target_length = 64

def tokenize_function(examples):
    model_inputs = tokenizer(examples["input_text"], max_length=max_input_length, truncation=True)
    labels = tokenizer(examples["target_text"], max_length=max_target_length, truncation=True)
    model_inputs["labels"] = [
        [(l if l != tokenizer.pad_token_id else -100) for l in label]
        for label in labels["input_ids"]
    ]
    return model_inputs

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_val = val_dataset.map(tokenize_function, batched=True)


In [ ]:
bleu = BLEU()

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

bleu_metric = evaluate.load("bleu")
chrf_metric = evaluate.load("chrf")

def compute_metrics(eval_preds):
    predictions, labels = eval_preds

    if isinstance(predictions, tuple):
        predictions = predictions[0]

    predictions = np.array(predictions)
    if predictions.ndim > 2:
        predictions = np.argmax(predictions, axis=-1)

    predictions = np.clip(predictions, 0, tokenizer.vocab_size - 1)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    preds_text = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    labels_text = tokenizer.batch_decode(labels, skip_special_tokens=True)

    em_scores = [int(pred.strip() == ref.strip()) for pred, ref in zip(preds_text, labels_text)]
    exact_match = np.mean(em_scores)

    bleu_score = bleu.corpus_score(preds_text, [labels_text]).score

    chrf_result = chrf_metric.compute(predictions=preds_text, references=labels_text)
    chrf_score = chrf_result["score"]

    return {
        "exact_match": exact_match,
        "bleu": bleu_score,
        "chrF": chrf_score
    }


In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir="/content/drive/MyDrive/translated_data/mt5-te-qa-checkpoints",
    label_smoothing_factor=0.1,
    learning_rate=3e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    weight_decay=0.01,
    save_total_limit=2,
    save_strategy="epoch",
    num_train_epochs=50,
    predict_with_generate=True,
    logging_dir="/content/drive/MyDrive/translated_data/logs",
    logging_steps=50,
    save_safetensors=False
)

In [ ]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

final_dir = "/content/drive/MyDrive/translated_data/week39-model"

os.makedirs(final_dir, exist_ok=True)
model.save_pretrained(final_dir)
tokenizer.save_pretrained(final_dir)


Step,Training Loss
50,26.358300
100,21.150400
150,18.574000
200,16.987800
250,14.583300
300,12.796100
350,11.141000
400,10.134800
450,9.686300
500,8.809900


In [ ]:
results = trainer.evaluate()
print(results)

In [ ]:
n_train = len(tokenized_train)
n_val = len(tokenized_val)

print(f"Training set size: {n_train}")
print(f"Validation set size: {n_val}")

n_val_answerable = sum(tokenized_val['answerable'])
n_val_unanswerable = n_val - n_val_answerable

print(f"Validation Answerable:   {n_val_answerable}")
print(f"Validation Unanswerable: {n_val_unanswerable}")

n_train_answerable = sum(tokenized_train['answerable'])
n_train_unanswerable = n_train - n_train_answerable

print(f"Training Answerable:   {n_train_answerable}")
print(f"Training Unanswerable: {n_train_unanswerable}")

val_answerable = tokenized_val.filter(lambda x: x["answerable"] == True)
val_unanswerable = tokenized_val.filter(lambda x: x["answerable"] == False)

results_answerable = trainer.evaluate(eval_dataset=val_answerable)
results_unanswerable = trainer.evaluate(eval_dataset=val_unanswerable)

print("Answerable:", results_answerable)
print("Unanswerable:", results_unanswerable)

## Week 41

In [ ]:
output_path = "/content/drive/MyDrive/translated_data/"
test_path = os.path.join(output_path, "test.json")
df_test = pd.read_json(test_path)

load_dir = "/content/drive/MyDrive/translated_data/week39-model"
tokenizer = AutoTokenizer.from_pretrained(load_dir)
model = AutoModelForSeq2SeqLM.from_pretrained(load_dir)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

da_en_name = "Helsinki-NLP/opus-mt-da-en"
da_en_tokenizer = AutoTokenizer.from_pretrained(da_en_name)
da_en_model = AutoModelForSeq2SeqLM.from_pretrained(da_en_name).to(device)

mbart_name = "facebook/mbart-large-50-many-to-many-mmt"
mbart_tokenizer = MBart50TokenizerFast.from_pretrained(mbart_name)
mbart_model = MBartForConditionalGeneration.from_pretrained(mbart_name).to(device)

def translate_da_to_en(text, model=da_en_model, tokenizer=da_en_tokenizer):
    """Translate Danish → English using OPUS-MT"""
    if not isinstance(text, str) or not text.strip():
        return ""
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=256).to(device)
    translated_tokens = model.generate(
        **inputs,
        num_beams=5,
        max_length=256,
        early_stopping=True
    )
    return tokenizer.batch_decode(translated_tokens, skip_special_tokens=True)[0]

def mbart_translate(text, src_lang, tgt_lang, model=mbart_model, tokenizer=mbart_tokenizer):
    """Generic MBART translation (used for English → Telugu)"""
    if not isinstance(text, str) or not text.strip():
        return ""
    tokenizer.src_lang = src_lang
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=256).to(device)
    translated_tokens = model.generate(
        **inputs,
        forced_bos_token_id=tokenizer.lang_code_to_id[tgt_lang],
        num_beams=5,
        max_length=256,
        early_stopping=True
    )
    return tokenizer.batch_decode(translated_tokens, skip_special_tokens=True)[0]

def translate_da_to_te(text):
    """Translate Danish → Telugu (via English pivot)"""
    intermediate_en = translate_da_to_en(text)
    final_te = mbart_translate(intermediate_en, src_lang="en_XX", tgt_lang="te_IN")
    return final_te

df_test = df_test[df_test["answer"].notnull()].reset_index(drop=True)

tqdm.pandas(desc="Translating questions to Telugu")
df_test["question_te"] = df_test["question"].progress_apply(translate_da_to_te)

tqdm.pandas(desc="Translating answers to Telugu")
df_test["answer_te"] = df_test["answer"].progress_apply(translate_da_to_te)

def get_question_context(row):
    return row["question"] + " " + row["context"]

def get_question_context_te(row):
    return row["question_te"] + " " + row["context"]

df_test["input_text"] = df_test.apply(get_question_context, axis=1)
df_test["input_text_te"] = df_test.apply(get_question_context_te, axis=1)
df_test["target_text"] = df_test["answer_inlang"]
df_test["target_text_te"] = df_test["answer_te"]

max_input_length = 768
max_target_length = 64

def tokenize_function(examples):
    model_inputs = tokenizer(examples["input_text"], max_length=max_input_length, truncation=True)
    labels = tokenizer(examples["target_text"], max_length=max_target_length, truncation=True)
    model_inputs["labels"] = [
        [(l if l != tokenizer.pad_token_id else -100) for l in label]
        for label in labels["input_ids"]
    ]
    return model_inputs

def tokenize_function_te(examples):
    model_inputs = tokenizer(examples["input_text_te"], max_length=max_input_length, truncation=True)
    labels = tokenizer(examples["target_text_te"], max_length=max_target_length, truncation=True)
    model_inputs["labels"] = [
        [(l if l != tokenizer.pad_token_id else -100) for l in label]
        for label in labels["input_ids"]
    ]
    return model_inputs

test_dataset = Dataset.from_pandas(df_test[["input_text", "target_text"]])
test_dataset_te = Dataset.from_pandas(df_test[["input_text_te", "target_text_te"]])

tokenized_test = test_dataset.map(tokenize_function, batched=True)
tokenized_test_te = test_dataset_te.map(tokenize_function_te, batched=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

source.spm:   0%|          | 0.00/820k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/788k [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/649 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/2.44G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/261 [00:00<?, ?B/s]

Translating questions to Telugu:   0%|          | 0/20 [00:00<?, ?it/s]

In [ ]:
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

bleu = BLEU()
bleu_metric = evaluate.load("bleu")
chrf_metric = evaluate.load("chrf")

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

def compute_metrics(eval_preds):
    predictions, labels = eval_preds

    if isinstance(predictions, tuple):
        predictions = predictions[0]

    predictions = np.array(predictions)
    if predictions.ndim > 2:
        predictions = np.argmax(predictions, axis=-1)

    predictions = np.clip(predictions, 0, tokenizer.vocab_size - 1)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    preds_text = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    labels_text = tokenizer.batch_decode(labels, skip_special_tokens=True)

    preds_text = [p.strip() for p in preds_text]
    labels_text = [l.strip() for l in labels_text]

    em_scores = [int(pred == ref) for pred, ref in zip(preds_text, labels_text)]
    exact_match = float(np.mean(em_scores))

    bleu_score = bleu.corpus_score(preds_text, [labels_text]).score

    chrf_result = chrf_metric.compute(predictions=preds_text, references=labels_text)
    chrf_score = chrf_result.get("score", 0.0)

    return {
        "exact_match": exact_match,
        "bleu": bleu_score,
        "chrF": chrf_score
    }

from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

eval_args = Seq2SeqTrainingArguments(
    output_dir="./eval_results",
    per_device_eval_batch_size=8,
    predict_with_generate=True,
    logging_dir="./logs_eval")

trainer = Seq2SeqTrainer(
    model=model,
    args=eval_args,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics)

print("Evaluating on Danish test set...")
results_da = trainer.evaluate(eval_dataset=tokenized_test)
print("Results (Danish):", results_da)

print("Evaluating on Telugu-translated test set...")
results_te = trainer.evaluate(eval_dataset=tokenized_test_te)
print("Results (Telugu):", results_te)

df_test.to_csv(os.path.join(output_path, "test_with_telugu.csv"), index=False, encoding="utf-8-sig")
print("Saved translated test set with Telugu translations.")

In [ ]:
sample_df = df_test.sample(5, random_state=42).reset_index(drop=True)

model.eval()
for i, row in sample_df.iterrows():
    inputs = tokenizer(
        row["input_text"],
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=768
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=64,
            num_beams=5,
            early_stopping=True
        )

    pred_answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

    print(f"Example {i+1}")
    print(f"Question (Danish): {row['question']}")
    print(f"Reference Answer (Danish): {row['answer_inlang']}")
    print(f"Model Prediction: {pred_answer}")
    print("-" * 120)
    print(f"Example {i+1}")
    print(f"Question (Telugu): {row['question_te']}")
    print(f"Reference Answer (Telugu): {row['answer_te']}")
    print(f"Model Prediction: {pred_answer}")
    print("-" * 120)


In [ ]:
if "answerable" not in df_test.columns:
    raise ValueError("The test set must include an 'answerable' column (True/False).")

n_test = len(df_test)
n_answerable = df_test["answerable"].sum()
n_unanswerable = n_test - n_answerable

print(f"Total test examples:     {n_test}")
print(f"Answerable examples:     {n_answerable}")
print(f"Unanswerable examples:   {n_unanswerable}")

test_answerable = Dataset.from_pandas(df_test[df_test["answerable"] == True][["input_text", "target_text"]])
test_unanswerable = Dataset.from_pandas(df_test[df_test["answerable"] == False][["input_text", "target_text"]])

test_answerable_te = Dataset.from_pandas(df_test[df_test["answerable"] == True][["input_text_te", "target_text_te"]])
test_unanswerable_te = Dataset.from_pandas(df_test[df_test["answerable"] == False][["input_text_te", "target_text_te"]])

tokenized_test_answerable = test_answerable.map(tokenize_function, batched=True)
tokenized_test_unanswerable = test_unanswerable.map(tokenize_function, batched=True)
tokenized_test_answerable_te = test_answerable_te.map(tokenize_function_te, batched=True)
tokenized_test_unanswerable_te = test_unanswerable_te.map(tokenize_function_te, batched=True)

print("Evaluating on Danish test subsets")
results_da_all = trainer.evaluate(eval_dataset=tokenized_test)
results_da_answerable = trainer.evaluate(eval_dataset=tokenized_test_answerable)
results_da_unanswerable = trainer.evaluate(eval_dataset=tokenized_test_unanswerable)

print("Results (Danish - All):", results_da_all)
print("Results (Danish - Answerable):", results_da_answerable)
print("Results (Danish - Unanswerable):", results_da_unanswerable)

print("Evaluating on Telugu test subsets")
results_te_all = trainer.evaluate(eval_dataset=tokenized_test_te)
results_te_answerable = trainer.evaluate(eval_dataset=tokenized_test_answerable_te)
results_te_unanswerable = trainer.evaluate(eval_dataset=tokenized_test_unanswerable_te)

print("Results (Telugu - ALL):", results_te_all)
print("Results (Telugu - Answerable):", results_te_answerable)
print("Results (Telugu - Unanswerable):", results_te_unanswerable)